## L = 3

Possible configurations: $ |\sigma\sigma\sigma\rangle, |\sigma\mu\sigma\rangle, |\mu\sigma\mu\rangle, |\mu\mu\mu\rangle $

In [1]:
import numpy as np
from numpy.linalg import eigh

In [2]:
def ising_fusCat_hamiltonian(r, theta):

    H = np.zeros(shape=(15, 15))

    # Elements
    H[0, 1:7] = -np.cos(theta)/np.sqrt(2)
    H[1:7, 0] = -np.cos(theta)/np.sqrt(2)
    for i in range(1, 13):
        H[i, i] = -2*r

    # The sin's
    H[7, 1] = H[7, 3] = -np.sin(theta)
    H[1, 7] = H[3, 7] = -np.sin(theta)

    H[8, 2] = H[8, 4] = -np.sin(theta)
    H[2, 8] = H[4, 8] = -np.sin(theta)

    H[9, 1] = H[9, 5] = -np.sin(theta)
    H[1, 9] = H[5, 9] = -np.sin(theta)

    H[10, 2] = H[10, 6] = -np.sin(theta)
    H[2, 10] = H[6, 10] = -np.sin(theta)

    H[11, 3] = H[11, 5] = -np.sin(theta)
    H[3, 11] = H[5, 11] = -np.sin(theta)

    H[12, 4] = H[12, 6] = -np.sin(theta)
    H[4, 12] = H[6, 12] = -np.sin(theta)

    # The cos's
    H[13, 7] = H[13, 9] = H[13, 11] = -np.cos(theta)
    H[7, 13] = H[9, 13] = H[11, 13] = -np.cos(theta)

    H[14, 8] = H[14, 10] = H[14, 12] = -np.cos(theta)
    H[8, 14] = H[10, 14] = H[12, 14] = -np.cos(theta)

    return H

In [3]:
H = ising_fusCat_hamiltonian(r=20.0, theta=np.pi/4)
energies, eigenvectors = eigh(H)

In [4]:
sorted_index = np.argsort(energies)
abs_tol = 0.0001

energy_deg_list = []
for index in sorted_index:
    if index == 0:
        energy_deg_list.append({"Energy": energies[0], "Degeneracy": 1})

    elif abs(energies[index] - energy_deg_list[-1]["Energy"]) < abs_tol:
        energy_deg_list[-1]["Degeneracy"] += 1

    else:
        energy_deg_list.append({"Energy": energies[index], "Degeneracy": 1})
        
energy_deg_list

[{'Energy': np.float64(-41.45040139046841), 'Degeneracy': 1},
 {'Energy': np.float64(-41.432431169608606), 'Degeneracy': 1},
 {'Energy': np.float64(-40.70710678118658), 'Degeneracy': 4},
 {'Energy': np.float64(-39.292893218813475), 'Degeneracy': 4},
 {'Energy': np.float64(-38.62462176937349), 'Degeneracy': 1},
 {'Energy': np.float64(-38.60508049791434), 'Degeneracy': 1},
 {'Energy': np.float64(0.03618782809531236), 'Degeneracy': 1},
 {'Energy': np.float64(0.03751166752291992), 'Degeneracy': 1},
 {'Energy': np.float64(0.03883533174658452), 'Degeneracy': 1}]

## Code

In [1]:
import numpy as np
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh

# Constants for simple objects
I, PSI, SIGMA = 0, 1, 2

def is_valid_state(state):
    """Filter states based on the Z_2 grading constraint."""
    for i in range(len(state)):
        left, right = state[i], state[(i + 1) % len(state)]
        if (left == I and right == PSI) or (left == PSI and right == I):
            return False
    return True

def generate_representatives(L, m):
    """Generate orbit representatives and valid states for a specific momentum sector."""
    all_states = [np.array(list(np.base_repr(i, base=3).zfill(L)), dtype=int) for i in range(3**L)]
    valid_states = [s for s in all_states if is_valid_state(s)]
    
    visited = set()
    reps = []
    
    for state in valid_states:
        state_tuple = tuple(state)
        if state_tuple not in visited:
            orbit = [tuple(np.roll(state, shift)) for shift in range(L)]
            rep = min(orbit)
            periodicity = len(set(orbit))
            
            # Check for destructive interference in this k-sector
            if (m * periodicity) % L == 0:
                reps.append({'rep': rep, 'R': periodicity})
                
            visited.update(orbit)
            
    return reps

def find_representative_and_shift(state):
    """
    Returns the orbit representative and the number of left shifts 
    required to reach it from the original state.
    """
    L = len(state)
    orbit = [tuple(np.roll(state, shift)) for shift in range(L)]
    rep = min(orbit)
    shift_j = orbit.index(rep)
    return rep, shift_j

def solve_spectrum(L, r, theta, num_eigenvalues=5):
    """Solve lowest eigenvalues for all momentum sectors."""
    spectra = {}
    for m in range(L):
        k = 2 * np.pi * m / L
        reps = generate_representatives(L, m)
        
        if not reps:
            continue
            
        H_k = build_hamiltonian_block(L, k, reps, r, theta)
        
        if H_k.shape[0] > num_eigenvalues:
            evals, _ = eigsh(H_k, k=num_eigenvalues, which='SA')
        else:
            evals = np.linalg.eigvalsh(H_k.toarray())
            
        spectra[k] = evals
    return spectra

def is_mu(x):
    """Helper to identify if a site is in the vacuum sector {I, psi} -> {0, 1}"""
    return x in (0, 1)

def is_sigma(x):
    """Helper to identify if a site is an anyon {sigma} -> {2}"""
    return x == 2

def build_hamiltonian_block(L, k, reps, r, theta):
    """
    Constructs the Hamiltonian matrix for a specific momentum sector k.
    State representation: 0 = I, 1 = psi, 2 = sigma
    """
    dim = len(reps)
    H_k = lil_matrix((dim, dim), dtype=complex)
    rep_to_idx = {reps[idx]['rep']: idx for idx in range(dim)}
    
    # Outer loop: Iterate over all representative states in this k-sector
    for i, state_data in enumerate(reps):
        rep = state_data['rep']
        R_a = state_data['R']
        
        # Inner loop: Iterate over every lattice site j to apply local H_i
        for j in range(L):
            left = rep[(j - 1) % L]
            mid = rep[j]
            right = rep[(j + 1) % L]
            
            # ==========================================
            # 1. H_dw Evaluation (Equation 14)
            # ==========================================
            val_dw = 0.0
            
            if is_mu(left) and is_mu(mid) and is_sigma(right): val_dw = r
            elif is_sigma(left) and is_mu(mid) and is_mu(right): val_dw = r
            elif is_mu(left) and is_sigma(mid) and is_sigma(right): val_dw = r
            elif is_sigma(left) and is_sigma(mid) and is_mu(right): val_dw = r
            
            # Apply overall negative sign to the diagonal element
            H_k[i, i] -= val_dw 
            
            # ==========================================
            # 2. H_flip Evaluation (Equation 15)
            # ==========================================
            transitions = []
            
            # |mu, mu, mu> -> cos(theta) |mu, sigma, mu>
            if is_mu(left) and is_mu(mid) and is_mu(right):
                transitions.append((2, np.cos(theta)))
                
            # |mu, mu, sigma> -> sin(theta) |mu, sigma, sigma>
            elif is_mu(left) and is_mu(mid) and is_sigma(right):
                transitions.append((2, np.sin(theta)))
                
            # |mu, sigma, nu> -> delta_{mu,nu} cos(theta) |mu, mu, mu>
            elif is_mu(left) and is_sigma(mid) and is_mu(right):
                if left == right: # Enforces the Kronecker delta
                    transitions.append((left, np.cos(theta)))
                    
            # |sigma, mu, mu> -> sin(theta) |sigma, sigma, mu>
            elif is_sigma(left) and is_mu(mid) and is_mu(right):
                transitions.append((2, np.sin(theta)))
                
            # |mu, sigma, sigma> -> sin(theta) |mu, mu, sigma>
            elif is_mu(left) and is_sigma(mid) and is_sigma(right):
                transitions.append((left, np.sin(theta)))
                
            # |sigma, mu, sigma> -> [cos(theta)/sqrt(2)] |sigma, sigma, sigma>
            elif is_sigma(left) and is_mu(mid) and is_sigma(right):
                transitions.append((2, np.cos(theta) / np.sqrt(2)))
                
            # |sigma, sigma, mu> -> sin(theta) |sigma, mu, mu>
            elif is_sigma(left) and is_sigma(mid) and is_mu(right):
                transitions.append((right, np.sin(theta)))
                
            # |sigma, sigma, sigma> -> [cos(theta)/sqrt(2)] ( |sigma, 0, sigma> + |sigma, 1, sigma> )
            elif is_sigma(left) and is_sigma(mid) and is_sigma(right):
                amp = np.cos(theta) / np.sqrt(2)
                transitions.extend([(0, amp), (1, amp)])
                
            # Process all valid off-diagonal transitions
            for new_mid, amp in transitions:
                new_state = list(rep)
                new_state[j] = new_mid
                
                # Verify the Z_2 grading constraint isn't violated
                if is_valid_state(new_state):
                    new_rep, shift_j = find_representative_and_shift(new_state)
                    
                    if new_rep in rep_to_idx:
                        b_idx = rep_to_idx[new_rep]
                        R_b = reps[b_idx]['R']
                        
                        # Apply overall negative sign, transition amplitude, and momentum phase factor
                        matrix_element = -amp * np.sqrt(R_a / R_b) * np.exp(1j * k * shift_j)
                        H_k[b_idx, i] += matrix_element

    return H_k.tocsr()

In [3]:
solve_spectrum(L=8, r=-20, theta=np.pi/4, num_eigenvalues=10)

{0.0: array([-82.82641449, -82.79138628, -82.10277402, -81.53956831,
        -81.45201996, -80.51205042, -80.06137129, -80.02498439,
        -80.01877125, -80.00366543]),
 1.0471975511965976: array([-82.30056427, -82.12318488, -81.82920057, -80.83628892,
        -80.65644533, -80.49650301, -80.03613505, -80.0210079 ,
        -80.00008517, -80.00579332]),
 2.0943951023931953: array([-82.08128632, -82.10967672, -81.87028085, -81.41216444,
        -81.31357046, -81.03005875, -80.85807618, -80.74479636,
        -80.36657522, -80.04373421]),
 3.141592653589793: array([-81.65564762, -81.43238297, -81.30656296, -81.13880655,
        -80.5411961 , -80.07246533, -80.07015747, -80.04996672,
        -80.00483751, -80.00252654]),
 4.1887902047863905: array([-82.10967672, -82.08128632, -81.87028085, -81.41216444,
        -81.31357046, -81.03005875, -80.85807618, -80.74479636,
        -80.36657522, -80.04373421]),
 5.235987755982989: array([-82.30056427, -82.12318488, -81.82920057, -80.83628892,
   

In [ ]:
# Find all degenerate ground states (within tol) across momentum sectors and display reconstructed wavefunctions
tol = 1e-4
amp_tol = 1e-6

L = 8
r = 20.0
theta = np.pi / 4

# get list of all valid states in a fixed order
all_states = [tuple(np.array(list(np.base_repr(i, base=3).zfill(L)), dtype=int)) for i in range(3**L)]
valid_states = [s for s in all_states if is_valid_state(list(s))]

# first pass: gather all eigenvalues to determine global ground energy
eig_info = []  # list of (m, k, evals, evecs, reps)
for m in range(L):
    k = 2 * np.pi * m / L
    reps = generate_representatives(L=L, m=m)
    if not reps:
        continue
    H_block = build_hamiltonian_block(L=L, k=k, reps=reps, r=r, theta=theta)
    H_dense = H_block.toarray()
    evals, evecs = np.linalg.eigh(H_dense)
    eig_info.append((m, k, evals, evecs, reps))

all_evals = np.hstack([info[2] for info in eig_info]) if eig_info else np.array([])
if all_evals.size == 0:
    print("No states found.")
else:
    ground_energy = np.min(all_evals)
    print(f"Global ground energy = {ground_energy:.8f} (tol = {tol})\n")

    degenerate_list = []
    for m, k, evals, evecs, reps in eig_info:
        for idx, E in enumerate(evals):
            if abs(E - ground_energy) <= tol:
                degenerate_list.append((m, k, E, evecs[:, idx], reps))

    for i, (m, k, E, vec, reps) in enumerate(degenerate_list):
        print(f"Degenerate GS #{i+1}: sector m={m}, k={k:.6f}, E={E:.8f}")
        # reconstruct full wavefunction on the valid_states basis
        psi_full = {s: 0+0j for s in valid_states}
        for b_idx, rep_data in enumerate(reps):
            rep = tuple(rep_data['rep'])
            Rb = rep_data['R']
            coeff = vec[b_idx]
            for t in range(Rb):
                s = tuple(np.roll(np.array(rep), t))
                amp = coeff * (1/np.sqrt(Rb)) * np.exp(1j * k * t)
                psi_full[s] += amp

        # list components in the order of valid_states
        amps = np.array([psi_full[s] for s in valid_states])
        norm = np.sum(np.abs(amps)**2)
        print(f"  Norm check: {norm:.12f}")

        # print significant components
        print("  Significant components (state -> amplitude):")
        for s, a in zip(valid_states, amps):
            if abs(a) > amp_tol:
                print("   ", "".join(map(str, s)), "->", a)
        print()
        

  Significant components (state -> amplitude, weight):
    22112200 -> (-0.32623110576843756-0.13512934847725638j) (weight=1.246866751908e-01)
    00221122 -> (-0.13512934847725622+0.3262311057684376j) (weight=1.246866751908e-01)
    02211220 -> (-0.3262311057684354+0.1351293484772615j) (weight=1.246866751908e-01)
    11220022 -> (0.13512934847725594-0.3262311057684377j) (weight=1.246866751908e-01)
    12200221 -> (0.32623110576843606-0.13512934847726005j) (weight=1.246866751908e-01)
    21122002 -> (-0.13512934847726074-0.3262311057684357j) (weight=1.246866751908e-01)
    22002211 -> (0.32623110576843767+0.13512934847725608j) (weight=1.246866751908e-01)
    20022112 -> (0.13512934847726019+0.32623110576843595j) (weight=1.246866751908e-01)
    02221220 -> (-0.005771361631877303+0.002398023842979846j) (weight=3.905913343741e-05)
    21220022 -> (0.002385310025762245-0.005776627867398365j) (weight=3.905913343741e-05)
    22122002 -> (-0.0023980238429798335-0.0057713616318773074j) (weight

## Draft calculations

In [174]:
# np.roll (a: np.array, shift: integer, axis: None)
# If shift > 0, then shift to the right, if shift < 0, then to the left

example_array = [1, 2, 3, 4, 5]
np.roll(example_array, -1)

# np.base_repr (number: integer, base: integer, padding: 0)
# input a number and returns a string
# padding is the number of zeros you want to add to the left of the string

np.base_repr(number=2, base=2, padding=1)
np.base_repr(number=10, base=3, padding=0)

# .zfill (L: integer)
# a build-in python method for padding zeros until a length L is reached

np.base_repr(number=10, base=3, padding=0).zfill(10)

'0000000101'

In [175]:
# E.g. L = 3
# The possible values of k = 0, 2pi/3, 4pi/3

L = 3
reps_dict = {}

for k in [0, 2*np.pi/3, 4*np.pi/3]:
    rep = generate_representatives(L, k)
    reps_dict[f"k = {k:.2f}"] = rep

reps_dict

{'k = 0.00': [{'rep': (np.int64(0), np.int64(0), np.int64(0)), 'R': 1},
  {'rep': (np.int64(0), np.int64(0), np.int64(2)), 'R': 3},
  {'rep': (np.int64(0), np.int64(2), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(1), np.int64(1)), 'R': 1},
  {'rep': (np.int64(1), np.int64(1), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(2), np.int64(2)), 'R': 3},
  {'rep': (np.int64(2), np.int64(2), np.int64(2)), 'R': 1}],
 'k = 2.09': [{'rep': (np.int64(0), np.int64(0), np.int64(2)), 'R': 3},
  {'rep': (np.int64(0), np.int64(2), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(1), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(2), np.int64(2)), 'R': 3}],
 'k = 4.19': [{'rep': (np.int64(0), np.int64(0), np.int64(2)), 'R': 3},
  {'rep': (np.int64(0), np.int64(2), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(1), np.int64(2)), 'R': 3},
  {'rep': (np.int64(1), np.int64(2), np.int64(2)), 'R': 3}]}

In [25]:
import time as time

# Start the timer
start_time = time.perf_counter()

# The code here
for i in range(100000000):
    pass

# End the timer
end_time = time.perf_counter()

print(f"{end_time - start_time}")

5.435733899939805


In [27]:
16 * np.cos(np.pi/4)**2 / (4 * 20) * (1 - 2/np.pi)

np.float64(0.03633802276324187)

In [28]:
-0.16414073--0.20006254

0.03592181

# Hand coded

In [1]:
L = 3

In [ ]:
for eig_U_psi in [1, -1]: # First, split with U(Ψ) into two blocks
    # Find the valid states

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.linalg as la
from itertools import product

class IsingAnyonChain:
    def __init__(self, L):
        self.L = L
        # 1. Generate all valid configuration states
        self.valid_states = self._generate_valid_states()
        self.N_full = len(self.valid_states)
        self.state_to_idx = {s: i for i, s in enumerate(self.valid_states)}
        
        # 2. Build the full Hamiltonian and U(sigma) matrices
        self.H_full = self._build_hamiltonian_full()
        self.U_sigma_full = self._build_U_sigma_full()

    def _generate_valid_states(self):
        """
        TODO: Replace with your exact Ising fusion rule logic.
        For demonstration, this generates a subset of valid periodic chains.
        """
        states = []
        for s in product([0, 1, 2], repeat=self.L):
            # Example constraint: 2s must not be adjacent to 2s (just a placeholder constraint)
            if not any(s[i] == 2 and s[(i+1)%self.L] == 2 for i in range(self.L)):
                states.append(s)
        return states

    def apply_U_psi(self, state):
        """Phase 1: Flips 0 <-> 1, leaves 2 unchanged."""
        return tuple(1 if x == 0 else (0 if x == 1 else 2) for x in state)

    def apply_U_sigma(self, state):
        """
        TODO: Implement Eq (20) from your paper here.
        Returns a list of tuples: [(amplitude, new_state), ...]
        """
        # --- PLACEHOLDER ---
        # To make the script runnable, we return a dummy diagonal action 
        # that mimics having eigenvalues of +sqrt(2) and -sqrt(2)
        dummy_sign = 1 if sum(state) % 2 == 0 else -1
        return [(dummy_sign * np.sqrt(2), state)]

    def _build_hamiltonian_full(self):
        """
        TODO: Implement your local Hamiltonian interaction here.
        Returning a dummy sparse Hermitian matrix for demonstration.
        """
        # Identity matrix as a placeholder
        return sp.eye(self.N_full, format='csc', dtype=complex)

    def _build_U_sigma_full(self):
        """Constructs the sparse U(sigma) operator in the full basis."""
        rows, cols, vals = [], [], []
        for i, s in enumerate(self.valid_states):
            for amp, s_out in self.apply_U_sigma(s):
                if s_out in self.state_to_idx:
                    rows.append(self.state_to_idx[s_out])
                    cols.append(i)
                    vals.append(amp)
        return sp.csc_matrix((vals, (rows, cols)), shape=(self.N_full, self.N_full), dtype=complex)

    def build_k_psi_projector(self, k_idx, psi_sign):
        """
        Simultaneously block-diagonalizes U(psi) and Momentum.
        Builds a highly sparse projection matrix from the Full Basis to the Sector Basis.
        """
        rows, cols, vals = [], [], []
        visited = set()
        col_counter = 0

        for s in self.valid_states:
            if s in visited:
                continue
            
            s_bar = self.apply_U_psi(s)
            
            # Identify the full orbit to mark as visited
            orbit_s = [s[x:] + s[:x] for x in range(self.L)]
            orbit_s_bar = [s_bar[x:] + s_bar[:x] for x in range(self.L)]
            visited.update(orbit_s)
            visited.update(orbit_s_bar)
            
            # Construct the unnormalized superposition vector mathematically
            nonzero_entries = {}
            for x in range(self.L):
                phase = np.exp(1j * 2 * np.pi * k_idx * x / self.L)
                
                idx_s = self.state_to_idx[orbit_s[x]]
                nonzero_entries[idx_s] = nonzero_entries.get(idx_s, 0) + phase
                
                idx_s_bar = self.state_to_idx[orbit_s_bar[x]]
                nonzero_entries[idx_s_bar] = nonzero_entries.get(idx_s_bar, 0) + (psi_sign * phase)
                
            # Compute norm. If it perfectly cancels (destructive interference), norm is 0.
            norm_sq = sum(abs(amp)**2 for amp in nonzero_entries.values())
            
            # If the state survives interference, add it to our basis projector
            if norm_sq > 1e-10:
                norm = np.sqrt(norm_sq)
                for r_idx, amp in nonzero_entries.items():
                    if abs(amp) > 1e-10:
                        rows.append(r_idx)
                        cols.append(col_counter)
                        vals.append(amp / norm)
                col_counter += 1

        # Return the projector sparse matrix
        return sp.csc_matrix((vals, (rows, cols)), shape=(self.N_full, col_counter), dtype=complex)

    def solve_sector(self, k_idx):
        """
        Executes the full pipeline:
        1. Splits by U(psi) and Momentum.
        2. Solves the (1, -1, 0) irrep block directly.
        3. Splits the U(psi) = +1 block by U(sigma).
        4. Solves the two resulting (1, 1, sqrt2) and (1, 1, -sqrt2) irreps.
        """
        results = {}

        # ==========================================
        # STEP 1: The U(psi) = -1 Block 
        # (Contains ONLY the (1, -1, 0) irrep)
        # ==========================================
        P_minus = self.build_k_psi_projector(k_idx, psi_sign=-1)
        if P_minus.shape[1] > 0:
            H_minus = P_minus.conj().T @ self.H_full @ P_minus
            # Diagonalize directly (Dense is fine here since size is reduced)
            evals_minus = la.eigvalsh(H_minus.toarray())
            results["Irrep (1, -1, 0)"] = evals_minus
        else:
            results["Irrep (1, -1, 0)"] = []

        # ==========================================
        # STEP 2: The U(psi) = +1 Block 
        # (Contains BOTH (1, 1, sqrt2) and (1, 1, -sqrt2))
        # ==========================================
        P_plus = self.build_k_psi_projector(k_idx, psi_sign=+1)
        if P_plus.shape[1] > 0:
            H_plus = P_plus.conj().T @ self.H_full @ P_plus
            U_sigma_plus = P_plus.conj().T @ self.U_sigma_full @ P_plus
            
            # ==========================================
            # STEP 3: Split by U(sigma) using Eigendecomposition
            # ==========================================
            # Because U_sigma and H commute, we find the eigenvectors of U_sigma
            # and project H into those eigenvectors.
            sigma_evals, sigma_evecs = la.eigh(U_sigma_plus.toarray())
            
            # Mask to separate +sqrt(2) from -sqrt(2)
            # (Using 0 as the threshold is perfectly safe for +/- 1.414)
            mask_plus = sigma_evals > 0
            mask_minus = sigma_evals < 0
            
            V_sigma_plus = sigma_evecs[:, mask_plus]
            V_sigma_minus = sigma_evecs[:, mask_minus]
            
            # Project H into the final separated bases
            if V_sigma_plus.shape[1] > 0:
                H_final_plus = V_sigma_plus.conj().T @ H_plus.toarray() @ V_sigma_plus
                results["Irrep (1, 1, sqrt2)"] = la.eigvalsh(H_final_plus)
            else:
                results["Irrep (1, 1, sqrt2)"] = []
                
            if V_sigma_minus.shape[1] > 0:
                H_final_minus = V_sigma_minus.conj().T @ H_plus.toarray() @ V_sigma_minus
                results["Irrep (1, 1, -sqrt2)"] = la.eigvalsh(H_final_minus)
            else:
                results["Irrep (1, 1, -sqrt2)"] = []

        return results

# ==========================================
# Execution Example
# ==========================================
if __name__ == "__main__":
    L = 8
    ------


/（7802..0363
636363566692603.36
.3
.36+3



=7/4515478/

    =7=7=、47=/98）0）（=）（=    chain = IsingAnyonChain(L)
    print(f"Total Full Hilbert Space Size: {chain.N_full}")
    
    k = 0 # Looking at k=0 sector
    spectra = chain.solve_sector(k_idx=k)
    
    print(f"\n--- Results for Momentum Sector k={k} ---")
    for irrep_name, energies in spectra.items():
        print(f"\n{irrep_name} Matrix Dimension: {len(energies)}")
        # Print lowest 3 energies for proof of concept
        if len(energies) > 0:
            print(f"Lowest Energies: {np.round(energies[:3], 4)}")

Total Full Hilbert Space Size: 3104

--- Results for Momentum Sector k=0 ---

Irrep (1, -1, 0) Matrix Dimension: 194
Lowest Energies: [1. 1. 1.]

Irrep (1, 1, sqrt2) Matrix Dimension: 88
Lowest Energies: [1. 1. 1.]

Irrep (1, 1, -sqrt2) Matrix Dimension: 82
Lowest Energies: [1. 1. 1.]
